## 1. El Arcade Learning Environment (ALE)


*   ¿Qué es el Arcade Learning Environment (ALE) y qué problema resuelve? ¿Qué relación tiene con el emulador Stella y con los juegos originales de Atari 2600?

Es un framework para el desarrollo de agentes de IA para juegos de ATARI 2600 (ROMs), su base es un emulador de esta consola llamado 'Stella', manteniendo separada la emulación del juego del agente. Hace uso de Gymnasium, interfaces Python y C++ para interactuar con los juegos.

*   ¿Qué diferencias existen entre las variantes de un mismo juego en ALE, por ejemplo ALE/SpaceInvaders-v5 frente a versiones con sufijos como -ram o parámetros como frameskip, repeat_action_probability y full_action_space?

Estas variantes no afectan al juego sino como el agente percibe el entorno, el tiempo y su gestion, que acciones puede hacer y el factor random. Por ejemplo '-ram' el agente no ve la pantalla sino la informacion de estados de lo que está pasando.

*   ¿Por qué es importante el concepto de frame skipping (salto de frames) en los entornos de Atari, y cómo afecta la velocidad de simulación y el aprendizaje de un agente?

El frameskipping hace que los recursos se usen de forma óptima y aceleran el aprendizaje sin afectar la eficiencia, tomandi decisiones cada x cantidad de  frames.

*   Investigue brevemente el juego Space Invaders: mecánica básica, objetivo y cómo se traduce la puntuación del juego original a la señal de recompensa (reward) del entorno.

Juego shooter publicado en 1978, el jugador debe disparar a los aliens  moviendose horizotalmente a lo largo de la parte inferior de la pantalla. El objetivo del jugador es no permitir que los aliens toquen tierra. La recompensa consisteria en que cada vez que se mata a un alien, el puntaje sube, por lo que se daría una recompensa cada que sube.

## 2. Espacios de observación y acción en entornos Atari

*   Describa el espacio de observación por defecto de ALE/SpaceInvaders-v5 (imagen RGB de 210x160x3 píxeles, tipo Box) y compárelo con el espacio de observación de CartPole-v1 visto en el laboratorio anterior. ¿Qué implicaciones tiene trabajar con observaciones basadas en imágenes en vez de vectores de
bajo nivel?

 Las diferencias principales son que el de space invaders es una imagen RGB con dimensiones mientras que el cartpole es un vector de 1 dimensión. Los datos que reciben son pixeles de los sprites del juego y el otro recibe la velocidad, angulo y posición. La implicaciones de trabajar con imagenes es que la complejidad del aprendizaje aumenta. Esto debido a que la precepcion del entorno, al tener dimesiones es más 'definida' que un vector de 1 dimension .

*   Investigue la variante de observación en RAM (obs_type="ram"), que expone directamente los 128 bytes de memoria de la consola Atari 2600. ¿En qué casos podría preferirse sobre la observación en imagen?

 Lo podemos usar cuando necesitamos que sea más ágil y no consuma una gran cantidad de recursos, en este caso para evitar tener que procesar una cantidad enorme de pixeles.

*   Describa el espacio de acción de Space Invaders e investigue el significado de cada una de las acciones disponibles (por ejemplo, NOOP, FIRE, RIGHT, LEFT, RIGHTFIRE, LEFTFIRE).

 Tiene un total de 6 acciones. NOOP es no hacer nada, FIRE es disparar pero la posicion de la nave no cambia, RIGHT y LEFT es moverse en las direcciones correspondientes a cada una. RIGHTFIRE y LEFTFIRE son FIRE y la direccion correspondiente al mismo tiempo.



*   ¿Qué es el wrapper AtariPreprocessing de Gymnasium y qué transformaciones aplica típicamente (escala de grises, redimensionamiento a 84x84, frame skipping, recorte de recompensa)? ¿Qué es FrameStackObservation (o FrameStack) y por qué suele usarse junto con AtariPreprocessing en juegos de Atari?

 Este wrapper implementa las tecnicas de preprocesamiento comunes para la consola Atari 2600, principalmente usa No-op Reset (no. random de acciones random con el reinicio para evitar que se estanque un comportamiento), frameskipping, max-pooling (valor maximo de pixeles de los ultimos fotogramas para eliminar 'parpadeos' que se dan en la consola), termination on life loss (epsodio termina temporalmente cuando se pierde una vida asi el agente aprende a mantenerse vivo) y reduccion de escala y color (convierte los colores a una escala de grises y reduce el tamaño a 84x84 por defecto con fines de eficiencia)

## 3. Módulo de desarrollo: funciones para interactuar con ALE

In [2]:
!pip install gymnasium[atari] ale-py shimmy

In [3]:
import gymnasium as gym
import ale_py
from gymnasium.wrappers import RecordVideo
import os
import glob
import numpy as np

In [5]:
def entorno(nom_entorno, video_folder=None, episode_trigger=None,
            name_prefix='SpaceInvaders_', render_mode=None, **kwargs):
    if video_folder is not None and render_mode is None:
        render_mode = "rgb_array"

    env = gym.make(nom_entorno, render_mode=render_mode, **kwargs)

    if video_folder is not None:
        os.makedirs(video_folder, exist_ok=True)
        if episode_trigger is None:
            episode_trigger = lambda ep: True
        env = RecordVideo(
            env,
            video_folder=video_folder,
            episode_trigger=episode_trigger,
            name_prefix=name_prefix,
        )

    return env

def baseline(observation, env):
  return env.action_space.sample()


def episodio(env, func_agente, max_steps=10000, seed=None):
    if seed is not None:
        observation, info = env.reset(seed=seed)
    else:
        observation, info = env.reset()

    recompensa_total = 0.0
    pasos = 0
    terminated = False
    truncated = False

    while not (terminated or truncated) and pasos < max_steps:
        action = func_agente(observation, env)
        observation, reward, terminated, truncated, info = env.step(action)
        recompensa_total += float(reward)
        pasos += 1

    return {
        "pasos": pasos,
        "recompensa_total": recompensa_total,
        "terminated": bool(terminated),
        "truncated": bool(truncated),
    }

In [6]:
def video(nom_entorno, func_agente, video_folder, name_prefix="Agente", n_episodios=1, max_steps=1000, seed=None, **env_kwargs):
  os.makedirs(video_folder, exist_ok=True)

  #grabar epsociods
  episode_trigger = lambda ep: True

  env = entorno(
      nom_entorno,
      video_folder=video_folder,
      episode_trigger=episode_trigger,
      name_prefix=name_prefix,
      render_mode="rgb_array",
      **env_kwargs,
  )

  metricas = []
  try:
      for i in range(n_episodios):
          ep_seed = None if seed is None else seed + i
          resultado = episodio(
              env, baseline, max_steps=max_steps, seed=ep_seed
          )
          metricas.append(resultado)
  finally:

      env.close()

  patron = os.path.join(video_folder, f"{name_prefix}*.mp4")
  videos = sorted(glob.glob(patron))

  return {"videos": videos, "metricas": metricas}

In [8]:
#videos
NOMBRE_ENTORNO = "ALE/SpaceInvaders-v5"
VIDEO_FOLDER  = "videos"

resultado = video(
    nom_entorno=NOMBRE_ENTORNO,
    func_agente=baseline,
    video_folder=VIDEO_FOLDER,
    name_prefix="spaceinvaders_aleatorio",
    n_episodios=1,
    max_steps=10000,
    seed=42,
)

print("Video guardado:")
for v in resultado["videos"]:
    print("  -", v)

print("\nMetricas/episodio:")
for i, m in enumerate(resultado["metricas"], 1):
    print(f"  Episodio {i}: pasos={m['pasos']}, "
          f"recompensa_total={m['recompensa_total']:.1f}, "
          f"terminated={m['terminated']}, truncated={m['truncated']}")

/usr/local/lib/python3.13/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Video guardado:
  - videos/spaceinvaders_aleatorio-episode-0.mp4

Metricas/episodio:
  Episodio 1: pasos=363, recompensa_total=50.0, terminated=True, truncated=False
